# 3 · Search

⏱ about 8 minutes

An **embedding** turns a text into a long list of numbers, a vector, so that texts with similar
*meaning* get similar vectors, even when they share no words. `similarity()` below measures the
angle between two vectors: 1 means the same direction, 0 means unrelated.

![Two sentences about cats point the same way, a sentence about revenue points elsewhere](img/embeddings.svg)

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBED_MODEL = os.environ["EMBED_MODEL"]
llm = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
import numpy as np


def embed(texts):
    response = llm.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in response.data]


def similarity(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))


cat, kitten, revenue = embed([
    "The cat sits on the mat.",
    "A kitten is resting on a rug.",
    "Quarterly revenue grew by four percent.",
])
print("numbers per vector:", len(cat))
print("cat vs kitten: ", round(similarity(cat, kitten), 3))
print("cat vs revenue:", round(similarity(cat, revenue), 3))

Try your own sentences in the cell above. Then embed all chunks from notebook 2. Sending all 200 at
once makes one very large request, so we send 64 at a time.

In [ ]:
import json

chunks = json.load(open("chunks.json"))
for start in range(0, len(chunks), 64):
    batch = chunks[start : start + 64]
    for item, vector in zip(batch, embed([c["text"] for c in batch])):
        item["vector"] = vector

print(len(chunks), "chunks embedded")

## Qdrant

Comparing a question with every chunk one by one works for 200 chunks, not for millions. A
**vector database** stores vectors and finds the closest ones fast. We use [Qdrant](https://qdrant.tech).
`":memory:"` runs it inside Python with nothing to install; in production it runs as a server.

A **collection** is like a table. Each **point** is one chunk: its vector, plus a **payload**
with everything we want back (text, file, page).

![A Qdrant collection: one point per chunk, each with id, vector and payload](img/qdrant.svg)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)

question = "What were the lifetime-encoded beads loaded with?"
hits = db.query_points("papers", query=embed([question])[0], limit=5).points
for hit in hits:
    print(f"{hit.score:.3f}  {hit.payload['source']} p. {hit.payload['page']}")
    print("      ", hit.payload["text"][:150].replace("\n", " "), "\n")

The chatbot runs as a separate program, so it can't see this notebook's variables. We save the
chunks with their vectors, and the bot loads them into its own Qdrant.

In [ ]:
from pathlib import Path

Path("index.json").write_text(json.dumps(chunks))
print("saved", len(chunks), "chunks to index.json")

## A search engine in Chainlit

No LLM yet: the bot shows the five closest chunks and their scores.

In [ ]:
%%writefile app.py
import json
import os

import chainlit as cl
from dotenv import load_dotenv
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

chunks = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(chunks[0]["vector"]), distance=Distance.COSINE),
)
db.upsert(
    "papers",
    points=[
        PointStruct(id=i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text")})
        for i, c in enumerate(chunks)
    ],
)

async def search(question, limit=5):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    # TODO 1: fewer results, and only good ones.
    return db.query_points("papers", query=response.data[0].embedding, limit=limit).points


@cl.on_message
async def on_message(message: cl.Message):
    hits = await search(message.content)
    results = [
        f"**{h.payload['source']}, p. {h.payload['page']}** (score {h.score:.2f})\n"
        f"> {h.payload['text'][:300].replace(chr(10), ' ')}…"
        for h in hits
    ]
    await cl.Message(content="\n\n".join(results) or "Nothing found.").send()

In [ ]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

## Your turn

Edit the `%%writefile app.py` cell above, run it again, and try your change in the browser.

Ask a few questions in the browser, including one the papers can't answer, like
*What is the capital of France?* Search always returns something: the closest chunks, however far
away they are.

1. **Fewer, better results.** Show 3 results instead of 5, and leave out the ones that don't
   match. Compare the scores of a good question and of the France question, pick a cutoff in
   between, and pass it to `query_points` as `score_threshold=`. Ask about France again.

**Bonus: ask in German.** Ask *Womit waren die Beads bei Kage et al. beladen?* The papers are in
English. Do the right chunks still come back, and with what scores?

<details>
<summary><b>Solutions</b> (try first!)</summary>

**1**
```python
async def search(question, limit=3):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=[question])
    return db.query_points(
        "papers", query=response.data[0].embedding, limit=limit,
        score_threshold=0.3,  # real questions scored 0.33–0.73 here, France 0.17
    ).points
```

**Bonus** — in testing, Kage page 2 still came first, with a score of 0.51 instead of 0.73 for the
English question: the embedding model maps both languages close together, just not as close.
</details>